# 06 · Training at scale

**Story so far:** the corpus is processed, featurized, and scored. This chapter trains
Review Radar's **sentiment classifier** and answers the question every ML platform team
hits: when do you fan out with Union primitives, and when do you reach for a distributed
framework like **Ray**?

We'll train the same model three ways — a single task, a hyperparameter sweep on
Union primitives, and a Ray variant — and end with a decision framework.

**Learning goals**

1. Write a training task that produces a durable model artifact (the one chapter
   [07](./07-serving.ipynb) will serve)
2. Run a hyperparameter sweep with `flyte.map` + reusable containers
3. Run the same workload on an ephemeral Ray cluster via `plugin_config`
4. Decide *deliberately* between `asyncio.gather`, `flyte.map`+reuse, and Ray

> **Platform prerequisite for §3:** the Ray plugin must be enabled in the deployment
> (KubeRay + plugin config — [appendix A](./appendix/A-deployment-adaptation.md) →
> *Compute plugins*). Sections 1-2 need nothing special.

In [ ]:
import asyncio
from typing import List

import flyte

flyte.init_from_config()

## 1. The training task

TF-IDF + logistic regression on synthetic labeled reviews — deliberately small so it runs
anywhere, with the exact shape of a real trainer: featurize → fit → evaluate → **return
the model as a `File` artifact**. Chapter [07](./07-serving.ipynb) wires this artifact
into a serving app by *task name*, so keep the env name (`training`) and task name
(`train_model`) stable.

In [ ]:
ml_image = (
    flyte.Image.from_debian_base(name="workshop-train", python_version=(3, 12))
    .with_pip_packages("scikit-learn==1.5.2", "joblib==1.4.2", "unionai-reuse>=0.1.15")
)

train_env = flyte.TaskEnvironment(
    name="training",
    image=ml_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
)

POS = ["absolutely love this {p}", "solid {p}, works as advertised", "the {p} exceeded expectations"]
NEG = ["disappointed with the {p}", "terrible {p}, arrived broken", "the {p} stopped working, support ignored me"]
PRODUCTS = ["espresso machine", "trail shoes", "headphones", "air fryer"]


def make_training_data(n: int, seed: int) -> tuple:
    """Helper defined in a cell → ships with the pickled bundle (authoring rule 1)."""
    import random
    rng = random.Random(seed)
    texts, labels = [], []
    for _ in range(n):
        label = rng.random() < 0.5
        tpl = rng.choice(POS if label else NEG)
        texts.append(tpl.format(p=rng.choice(PRODUCTS)))
        labels.append(int(label))
    return texts, labels


@train_env.task(cache="auto")
async def train_model(n_examples: int = 2000, c: float = 1.0, seed: int = 7) -> flyte.io.File:
    import joblib
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import make_pipeline

    texts, labels = make_training_data(n_examples, seed)
    model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=c, max_iter=1000))
    score = cross_val_score(model, texts, labels, cv=3).mean()
    model.fit(texts, labels)
    print(f"C={c} cv_accuracy={score:.4f}")

    joblib.dump(model, "sentiment.joblib")
    return await flyte.io.File.from_local("sentiment.joblib")


run = await flyte.run.aio(train_model, n_examples=2000, c=1.0)
print(run.url)
await run.wait.aio()

> Real trainers swap the synthetic generator for the featurized corpus from
> [02](./02-data-flow.ipynb)-[05](./05-reusable-containers.ipynb) (passed as a
> `File`/`DataFrame` input) and, when the model needs it, add
> `gpu=flyte.GPU(device=..., quantity=...)` to the environment.

## 2. Hyperparameter sweep: Union primitives

A sweep is embarrassingly parallel — the same trainer over a list of configs. That's
`flyte.map` + a reusable pool, with every trial **individually cached, retried, and
visible in the UI**:

In [ ]:
from datetime import timedelta
from functools import partial

sweep_env = flyte.TaskEnvironment(
    name="sweep_driver",
    image=ml_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    depends_on=[train_env],
)


@train_env.task(cache="auto")
async def evaluate_config(c: float, n_examples: int, seed: int) -> float:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import make_pipeline

    texts, labels = make_training_data(n_examples, seed)
    model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=c, max_iter=1000))
    return float(cross_val_score(model, texts, labels, cv=3).mean())


@sweep_env.task
async def hyperparameter_sweep(cs: List[float], n_examples: int = 2000) -> dict:
    fn = partial(evaluate_config, n_examples=n_examples, seed=7)
    scores: List[float] = []
    async for r in flyte.map.aio(fn, cs, return_exceptions=True, group_name="trials"):
        if isinstance(r, Exception):
            raise r
        scores.append(r)
    best = max(zip(scores, cs))
    return {"best_c": best[1], "best_accuracy": round(best[0], 4)}


run = await flyte.run.aio(hyperparameter_sweep, cs=[0.01, 0.1, 1.0, 10.0])
print(run.url)
await run.wait.aio()
run.outputs()

> For big sweeps, add a `ReusePolicy` to `train_env` (chapter
> [05](./05-reusable-containers.ipynb)) so trials share warm pods, and `concurrency=1`
> since training is CPU-bound.

## 3. The same sweep on Ray

Union runs Ray natively: a task environment declares a `RayJobConfig`, and the platform
provisions a **transient, right-sized Ray cluster** (via KubeRay) for that execution,
then tears it down. The task body becomes the Ray driver.

What the integration gives you per execution: head/worker groups sized in code,
autoscaling (`min_replicas`/`max_replicas`), per-group resources (a GPU group and a CPU
group can coexist), Ray `runtime_env` passthrough, `shutdown_after_job_finishes` +
`ttl_seconds_after_finished` lifecycle, and `address=` to attach to an existing
long-lived cluster instead.

In [ ]:
from flyteplugins.ray import HeadNodeConfig, RayJobConfig, WorkerNodeConfig

ray_image = (
    flyte.Image.from_debian_base(name="workshop-ray", python_version=(3, 12))
    .with_pip_packages("ray[default]==2.46.0", "flyteplugins-ray==2.5.7",
                       "scikit-learn==1.5.2")
)

ray_config = RayJobConfig(
    head_node_config=HeadNodeConfig(requests=flyte.Resources(cpu="1", memory="2Gi")),
    worker_node_config=[
        WorkerNodeConfig(
            group_name="cpu-workers",
            replicas=2,
            min_replicas=1,
            max_replicas=4,
            requests=flyte.Resources(cpu="2", memory="4Gi"),
        ),
    ],
    enable_autoscaling=True,
    shutdown_after_job_finishes=True,
    ttl_seconds_after_finished=300,
)

ray_env = flyte.TaskEnvironment(
    name="ray_sweep",
    image=ray_image,
    plugin_config=ray_config,                     # <- this makes it a Ray task env
    resources=flyte.Resources(cpu="2", memory="4Gi"),   # the driver itself
)


@ray_env.task
async def ray_sweep(cs: List[float], n_examples: int = 2000) -> dict:
    """The task body is the Ray driver on the ephemeral cluster's head node."""
    import ray

    @ray.remote
    def trial(c: float) -> float:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import cross_val_score
        from sklearn.pipeline import make_pipeline
        texts, labels = make_training_data(n_examples, 7)
        model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=c, max_iter=1000))
        return float(cross_val_score(model, texts, labels, cv=3).mean())

    scores = ray.get([trial.remote(c) for c in cs])
    best = max(zip(scores, cs))
    return {"best_c": best[1], "best_accuracy": round(best[0], 4)}


# Requires the Ray plugin enabled in this deployment:
# run = await flyte.run.aio(ray_sweep, cs=[0.01, 0.1, 1.0, 10.0])

### Under the hood

Running the Ray cell makes the plugin create a `RayJob` custom resource; the KubeRay
operator spins up head/worker pods in the data plane, runs the driver, and cleans up
after the TTL. On BYOC/self-managed you can watch it:
`kubectl get rayjobs,pods -n <project>-<domain>`.

## 4. Decision framework

Both sweeps produced the same answer. The differences that matter:

| Question | If yes → |
|---|---|
| Do workers need to **talk to each other**? (allreduce, parameter servers, Ray Train/Tune/RLlib) | **Ray** |
| Do you need Ray's **shared-memory object store** (large arrays shared zero-copy)? | **Ray** |
| Are you migrating an **existing Ray codebase** as-is? | **Ray** (ephemeral clusters per run) |
| Is the work **embarrassingly parallel** trials/items? | **`flyte.map` + ReusePolicy** |
| Do you want **per-trial caching, retries, and UI visibility**? | **`flyte.map` + ReusePolicy** |
| Is it a moderate, **I/O-bound** fan-out inside one pod's capacity? | **`asyncio.gather`** |
| Do you mainly want **warm workers / model-in-memory** between items? | **ReusePolicy** |

Rules of thumb for the gray zone:

- **Hyperparameter sweeps** rarely need Ray — per-trial caching alone (re-run the sweep,
  completed trials are free) usually beats the ephemeral-cluster spin-up.
- **Batch inference** is chapter [05](./05-reusable-containers.ipynb), not a Ray Serve
  deployment.
- **Multi-node distributed training** (gradient sync) is genuinely Ray's home turf (or
  the PyTorch-elastic plugin) — keep it there, and let Flyte version/cache/schedule the
  job around it.
- **Operational cost is asymmetric:** reusable containers are a `ReusePolicy(...)` line;
  Ray means the KubeRay operator, its upgrades, and Ray-version compatibility become part
  of the platform surface.

A practical port heuristic: if the only Ray APIs in a file are `ray.remote` + `ray.get`
over independent inputs, it ports to `flyte.map` in an afternoon — §3 → §2 above is
exactly that port, in reverse.

> 💬 **Discuss:** inventory the customer's current or planned Ray workloads — which
> actually use collectives, Train/Tune, or the object store, and which are `ray.remote`
> fan-outs over independent inputs that would port to `flyte.map`?

**Story checkpoint:** `training.train_model` produced a versioned `sentiment.joblib`
artifact with a known accuracy. Time to put it behind an API.

## Further reading

- Next: [07-serving](./07-serving.ipynb) — the artifact goes live
- Union docs → Integrations → Ray